# 01 - Data preparation (Tomer only, run once)

**CPU notebook - do NOT attach a GPU** (saves your GPU quota; nothing here needs one).
Internet must be ON. Runtime ~15-25 min.

Builds the three artifacts every runner shares:

| File | What |
|---|---|
| `probes.json` | 3000 facts x 10 length-matched candidates, templates, train/val/test splits |
| `exposure.npz` | cumulative exposure per fact at every 1000-step bin, all 6 epochs |
| `facts.parquet` | the fact metadata for those 3000 |

Everyone must use the **same** probe set or the results cannot be pooled. The notebook prints a
`PROBE_HASH`; each runner recomputes it and asserts it matches.

## 1. Environment

In [ ]:
import sys, os, subprocess, json, time, gc, shutil, platform, hashlib, re

ON_KAGGLE = os.path.isdir("/kaggle")

def _writable(d):
    try:
        os.makedirs(d, exist_ok=True)
        t = os.path.join(d, ".wtest")
        with open(t, "w") as f: f.write("x")
        os.remove(t); return d
    except Exception:
        return None

SCRATCH = None
for cand in (["/kaggle/temp", "/tmp"] if ON_KAGGLE else ["./_scratch"]):
    SCRATCH = _writable(cand)
    if SCRATCH: break
assert SCRATCH, "no writable scratch directory"
OUT = _writable("/kaggle/working" if ON_KAGGLE else "./_out")
TMP = os.path.join(SCRATCH, "ckpt"); os.makedirs(TMP, exist_ok=True)
os.environ["HF_HOME"] = os.path.join(SCRATCH, "hf")     # must precede any HF import
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import transformers
    from packaging.version import parse as V
    if V(transformers.__version__) < V("4.47"):
        pipq("-U", "transformers>=4.47")
        os.execv(sys.executable, [sys.executable] + sys.argv)
except ImportError:
    pipq("-U", "transformers>=4.47")
import numpy as np, pandas as pd, transformers
try:
    import pyarrow
except ImportError:
    pipq("pyarrow"); import pyarrow

import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
except Exception as e:
    raise SystemExit("Internet is OFF. Kaggle -> Settings -> Internet -> On. " + str(e))
print("scratch:", TMP, "| out:", OUT, "| transformers:", transformers.__version__)

## 2. Config

In [ ]:
SEED         = 0
N_PER_BUCKET = 300          # 300 x 10 buckets = 3000 facts
N_CAND       = 10           # gold + 9 distractors -> chance = 0.100
GEN_TOKENS   = 10

BUCKET_EDGES  = [-1, 0, 1, 2, 4, 7, 13, 25, 60, 200, 10**9]
BUCKET_LABELS = ["0","1","2","3-4","5-7","8-13","14-25","26-60","61-200","200+"]

# Deterministic, hand-written. No LLM in the loop, so probe construction is reproducible.
TEMPLATES = {
    "director":       "The director of {s} is",
    "screenwriter":   "The screenwriter of {s} is",
    "genre":          "The genre of {s} is",
    "producer":       "The producer of {s} is",
    "author":         "The author of {s} is",
    "composer":       "The composer of {s} is",
    "country":        "{s} is located in the country of",
    "capital":        "The capital of {s} is",
    "place of birth": "{s} was born in the city of",
    "father":         "The father of {s} is",
    "sport":          "{s} plays the sport of",
    "occupation":     "The occupation of {s} is",
    "capital of":     "{s} is the capital of",
    "religion":       "The religion of {s} is",
    "mother":         "The mother of {s} is",
    "color":          "The color of {s} is",
}
STEPS_PER_EPOCH = 109672        # LMEnt: 1 epoch = 109,672 optimiser steps
TOKENIZER_REF   = ("dhgottesman/LMEnt-1B-6E", "step10000")   # identical across all LMEnt models

## 3. Load PopQA-KAS fact metadata

In [ ]:
from huggingface_hub import HfFileSystem
import pyarrow.parquet as pq

SCALAR_COLS = ["id","subj","prop","obj","subj_id","obj_id","s_pop","o_pop",
               "question","possible_answers",
               "subject_num_chunks","answer_num_chunks","num_shared_chunks"]

def load_popqa_kas(columns):
    """Column-selective remote parquet read: never downloads the huge list columns
       unless we ask for them."""
    fs = HfFileSystem(); parts=[]
    for i in range(9):
        p = f"datasets/dhgottesman/popqa-kas/data/train-0000{i}-of-00009.parquet"
        with fs.open(p, "rb") as fh:
            parts.append(pq.ParquetFile(fh).read(columns=columns).to_pandas())
        print(f"  shard {i}", flush=True)
    return pd.concat(parts, ignore_index=True)

t0 = time.time()
facts = load_popqa_kas(SCALAR_COLS)
print(f"{len(facts)} facts in {time.time()-t0:.0f}s")
assert len(facts) == 14267 and facts.prop.nunique() == 16
facts["bucket"] = pd.cut(facts.num_shared_chunks, BUCKET_EDGES, labels=BUCKET_LABELS)
print(facts.bucket.value_counts().reindex(BUCKET_LABELS).to_string())

## 4. Build the probe set

Distractors come from the same relation's object pool (type-consistent), are token-length-matched
to the gold (limits surface-form bias), and are screened against the gold's alias list.

`corpus_prior_idx` is the most corpus-frequent **distractor**. Defining the prior over *all*
candidates makes "did the model fall back on the prior?" structurally impossible to answer
whenever the prior happens to be the gold - which biases high-exposure buckets downward. The
pilot hit exactly that.

In [ ]:
def as_list(x):
    if x is None: return []
    return [str(v) for v in list(x)]

def build_probes(facts, tok, n_per_bucket=N_PER_BUCKET, seed=SEED):
    """Deterministic probe construction. Same seed -> byte-identical probe set."""
    def ntok(s): return len(tok(" " + s, add_special_tokens=False)["input_ids"])
    rng = np.random.default_rng(seed)
    sample = pd.concat([g.sample(min(n_per_bucket, len(g)), random_state=seed)
                        for _, g in facts.groupby("bucket", observed=True)]).reset_index(drop=True)
    pools    = {p: sorted(set(g.obj)) for p, g in facts.groupby("prop")}
    pool_len = {p: np.array([ntok(o) for o in pools[p]]) for p in pools}
    obj_freq = facts.groupby("obj").answer_num_chunks.max().to_dict()

    probes = []
    for r in sample.itertuples():
        pool, plens = pools[r.prop], pool_len[r.prop]
        gold_len = ntok(r.obj)
        banned   = {r.obj.lower()} | {a.lower() for a in as_list(r.possible_answers)}
        cands = [r.obj]
        for tol in (0, 1, 2, 99):                 # widen until we have enough
            idx = np.flatnonzero(np.abs(plens - gold_len) <= tol)
            rng.shuffle(idx)
            for j in idx:
                c = pool[j]
                if c.lower() in banned or c in cands: continue
                cands.append(c)
                if len(cands) == N_CAND: break
            if len(cands) == N_CAND: break
        if len(cands) < N_CAND: continue
        # Corpus prior among the DISTRACTORS only. Defining it over all candidates makes
        # chose_prior structurally impossible whenever the prior happens to be the gold,
        # which silently biases high-exposure buckets downward.
        dist_freq = [obj_freq.get(c, 0) for c in cands[1:]]
        probes.append({
            "fact_id": int(r.id), "subj": r.subj, "prop": r.prop, "obj": r.obj,
            "bucket": str(r.bucket), "n_shared": int(r.num_shared_chunks),
            "n_subject": int(r.subject_num_chunks), "subj_id": int(r.subj_id),
            "prompt": TEMPLATES[r.prop].format(s=r.subj),
            "candidates": cands,                       # index 0 is ALWAYS the gold
            "corpus_prior_idx": int(np.argmax(dist_freq)) + 1,   # never 0
            "possible_answers": as_list(r.possible_answers),
        })
    return probes

def probe_hash(probes):
    """Stable fingerprint. All four runners must report the same value."""
    payload = json.dumps([[p["fact_id"], p["candidates"]] for p in probes],
                         sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode()).hexdigest()[:16]

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(TOKENIZER_REF[0], subfolder=TOKENIZER_REF[1], use_fast=True)

t0 = time.time()
probes = build_probes(facts, tok)
PROBE_HASH = probe_hash(probes)
print(f"{len(probes)} probes in {time.time()-t0:.0f}s")
print("PROBE_HASH:", PROBE_HASH, " <-- every runner must report this exact value")

gl = np.array([len(tok(" "+p["candidates"][0], add_special_tokens=False)["input_ids"]) for p in probes])
dl = np.array([len(tok(" "+c, add_special_tokens=False)["input_ids"]) for p in probes for c in p["candidates"][1:]])
print(f"gold token len {gl.mean():.3f} vs distractor {dl.mean():.3f}")
assert abs(gl.mean()-dl.mean()) < 0.5, "distractor length matching failed"
assert all(p["corpus_prior_idx"] != 0 for p in probes), "prior must never be the gold index"
print("per-bucket:", pd.Series([p["bucket"] for p in probes]).value_counts().reindex(BUCKET_LABELS).to_dict())

## 5. Train / val / test split, grouped by entity

In [ ]:
# Split by SUBJECT ENTITY, not by fact: the same entity appearing in train and test
# would leak. Thresholds get tuned on val; headline numbers come from test only.
rng = np.random.default_rng(SEED)
subs = sorted({p["subj_id"] for p in probes})
rng.shuffle(subs)
n = len(subs); tr, va = int(.50*n), int(.75*n)
split_of = {}
for k, s in enumerate(subs):
    split_of[s] = "train" if k < tr else ("val" if k < va else "test")
for p in probes: p["split"] = split_of[p["subj_id"]]
sp = pd.Series([p["split"] for p in probes]).value_counts()
print("entities:", n, "| probes per split:", sp.to_dict())
ov = pd.crosstab(pd.Series([p["bucket"] for p in probes], name="bucket"),
                 pd.Series([p["split"] for p in probes], name="split"))
print(ov.reindex(BUCKET_LABELS).to_string())

## 6. Cumulative exposure

`batch_indices_epoch_{1..6}.npy` maps every pretraining chunk to the optimiser step that consumed
it. Combined with each fact's `shared_chunks`, this gives **how many co-occurrences the model had
actually seen by step t** - the real causal variable, rather than a static total.

In [ ]:
from huggingface_hub import hf_hub_download

# Which pretraining chunks does each fact live in, and when did the model see them?
want = {p["fact_id"] for p in probes}
t0 = time.time()
ch = load_popqa_kas(["id","shared_chunks","subject_chunks"])
ch = ch[ch.id.isin(want)].set_index("id")
print(f"chunk lists for {len(ch)} facts in {time.time()-t0:.0f}s")

BIN   = 1000
NBINS = STEPS_PER_EPOCH*6//BIN + 1
order = [p["fact_id"] for p in probes]
H_sh  = np.zeros((len(order), NBINS), dtype=np.int32)
H_su  = np.zeros((len(order), NBINS), dtype=np.int32)
covered = 0; total = 0

for ep in range(1, 7):
    f = hf_hub_download("dhgottesman/LMEnt-Dataset",
                        f"dataset-cache/batch_indices_epoch_{ep}.npy", repo_type="dataset")
    bi = np.load(f, allow_pickle=True)
    flat  = np.concatenate([np.asarray(b) for b in bi])
    stp   = np.repeat(np.arange(len(bi), dtype=np.int64), [len(b) for b in bi])
    N     = int(flat.max())+1
    first = np.full(N, -1, dtype=np.int64); first[flat] = stp
    base  = (ep-1)*STEPS_PER_EPOCH
    for i, fid in enumerate(order):
        for col, H in (("shared_chunks", H_sh), ("subject_chunks", H_su)):
            c = np.asarray(ch.at[fid, col])
            if len(c) == 0: continue
            c = c[c < N]; s = first[c]; s = s[s >= 0]
            if len(s): np.add.at(H[i], (base + s)//BIN, 1)
            if ep == 1 and col == "shared_chunks":
                covered += len(s); total += len(np.asarray(ch.at[fid, col]))
    del bi, flat, stp, first; gc.collect()
    print(f"  epoch {ep} done", flush=True)

print(f"chunk->step coverage (epoch 1): {covered}/{total} = {covered/max(total,1):.3%}")
CUM_SH = H_sh.cumsum(1)      # cumulative exposure at each 1000-step bin
CUM_SU = H_su.cumsum(1)
print("cumulative exposure array:", CUM_SH.shape, f"{CUM_SH.nbytes/1e6:.1f} MB")
N_EPOCHS = 6
chk = CUM_SH[:, -1].astype(float)
ns  = np.array([p["n_shared"] for p in probes], dtype=float) * N_EPOCHS
ok  = np.isclose(chk, ns, rtol=0.02) | (ns == 0)
print(f"final cumulative exposure matches num_shared_chunks x {N_EPOCHS} (within 2%) for "
      f"{int(ok.sum())}/{len(ns)} facts")
assert ok.mean() > 0.95, "exposure accounting is off for too many facts"

## 7. Save and hand off

In [ ]:
payload = {"probe_hash": PROBE_HASH, "seed": SEED, "n_per_bucket": N_PER_BUCKET,
           "n_cand": N_CAND, "bucket_labels": BUCKET_LABELS, "bin": BIN,
           "steps_per_epoch": STEPS_PER_EPOCH, "templates": TEMPLATES, "probes": probes}
with open(os.path.join(OUT, "probes.json"), "w") as f:
    json.dump(payload, f)
np.savez_compressed(os.path.join(OUT, "exposure.npz"),
                    fact_id=np.array(order), cum_shared=CUM_SH, cum_subject=CUM_SU, bin=BIN)
facts[facts.id.isin(want)].to_parquet(os.path.join(OUT, "facts.parquet"), index=False)

print("=" * 72)
print("DATA PREP COMPLETE")
print("=" * 72)
print("PROBE_HASH:", PROBE_HASH)
for f in sorted(os.listdir(OUT)):
    print(f"  {f}  ({os.path.getsize(os.path.join(OUT,f))/1e6:.2f} MB)")
print("")
print("NEXT: create a Kaggle Dataset from probes.json + exposure.npz + facts.parquet,")
print("      name it  lment-probes , and share it with the other three runners.")
print("      Their notebooks attach it and assert PROBE_HASH matches.")